In [3]:
from operator import matmul
from xml.etree import ElementInclude
import pandas as pd
import numpy as np
from mass_charge_dict import ELEMENTS2Z, Z2ELEMENTS,elements_dict
from scipy import linalg
from math import log10 , floor
import os
import shutil
from functions import *
import pandas as pd

In [15]:
input_path_coord = 'tests/NH3/coord.xyz'
input_path_hess = 'tests/NH3/hessian'
input_path_dipm = 'tests/NH3/xyz_dipm.csv'

coord,head = import_coord(input_path_coord)
hessian = import_hess(input_path_hess,coord)
dipm = import_dipm(input_path_dipm)

dipm = dipm.iloc[:,:-3]
############
########### Rotation of coordinates and hessian into intermediate position
# Calculating center of mass 
s = center_mass(coord) 
# Translation of coordinate system
vec_trans(coord,s)

#vec_trans(dipm,s)
# Calculating moment of inertia
I = inert_tensor(coord)

# Calculating eigenvalues and eigenvectors 
eig_val,eig_vec = linalg.eigh(I)

# Check if the coordinate system is right-handed --> important for chirality

eig_vec = check_eig_vec(eig_vec)

# Rotating eigenvectors, so that highest values are positive

eig_vec = eig_vec_rot(eig_vec)

# Rotation of the coordinates and atomic dipole moments
coord = coord_rot(coord,eig_vec.copy())

dipm = coord_rot(dipm,eig_vec.copy())

# Construction of the rotation matrix of the hessian and the rotation
P = rotM_hess(eig_vec.copy(),coord)
############

hessian = matmul(matmul(P,hessian),np.transpose(P))

hess_OH = hessian[0:3,3:6]
R_euler = get_R_euler(coord,dipm,0,1)

coord_rot(coord,R_euler)
coord_rot(dipm,R_euler)

rotM_Z = rot_Z(1/2*np.pi)

print(hess_OH)
print(coord)
for i in range(2):
    coord_rot(coord,rotM_Z)
    hess_OH = matmul(matmul(rotM_Z,hess_OH),np.transpose(rotM_Z))
    print(hess_OH)
    print(coord)


[[-2.93376176e-01  2.24572773e-01  6.32100000e-07]
 [ 1.82035980e-01 -2.03830093e-01 -2.79700000e-07]
 [ 7.06400000e-07 -4.28200000e-07  0.00000000e+00]]
  atoms             x             y         z
0     O -2.941782e-17  5.551115e-17  0.480429
1     H  2.941782e-17 -5.551115e-17 -0.480429
2     H -1.015736e-16 -9.275341e-01  0.731289
[[-2.03830093e-01 -1.82035980e-01  2.79700000e-07]
 [-2.24572773e-01 -2.93376176e-01  6.32100000e-07]
 [ 4.28200000e-07  7.06400000e-07  0.00000000e+00]]
  atoms             x             y         z
0     O -5.551115e-17 -2.941782e-17  0.480429
1     H  5.551115e-17  2.941782e-17 -0.480429
2     H  9.275341e-01 -1.583687e-16  0.731289
[[-2.93376176e-01  2.24572773e-01 -6.32100000e-07]
 [ 1.82035980e-01 -2.03830093e-01  2.79700000e-07]
 [-7.06400000e-07  4.28200000e-07  0.00000000e+00]]
  atoms             x             y         z
0     O  2.941782e-17 -5.551115e-17  0.480429
1     H -2.941782e-17  5.551115e-17 -0.480429
2     H  2.151638e-16  9.275341e

In [2]:
molecule = 'H2O'
input_path_coord = f'tests/coord_only/{molecule}/start_coord/coord.xyz'
input_path_hess = f'tests/coord_only/{molecule}/start_coord/hessian'
input_path_dipm = f'tests/coord_only/{molecule}/start_coord/xyz_dipm.csv'

coord,head = import_coord(input_path_coord)
hessian = import_hess(input_path_hess,coord)
dipm = import_dipm(input_path_dipm)

dipm = dipm.iloc[:,:-3]
############
########### Rotation of coordinates and hessian into intermediate position
# Calculating center of mass 
s = center_mass(coord) 
# Translation of coordinate system
vec_trans(coord,s)

#vec_trans(dipm,s)
# Calculating moment of inertia
I = inert_tensor(coord)

# Calculating eigenvalues and eigenvectors 
eig_val,eig_vec = linalg.eigh(I)

# Check if the coordinate system is right-handed --> important for chirality

eig_vec = check_eig_vec(eig_vec)

# Rotating eigenvectors, so that highest values are positive

eig_vec = eig_vec_rot(eig_vec)

# Rotation of the coordinates and atomic dipole moments
coord = coord_rot(coord,eig_vec.copy())

dipm = coord_rot(dipm,eig_vec.copy())

# Construction of the rotation matrix of the hessian and the rotation
P = rotM_hess(eig_vec.copy(),coord)
############

hessian = matmul(matmul(P,hessian),np.transpose(P))

hess_OH = hessian[0:3,3:6]
R_euler = get_R_euler(coord,dipm,0,1)

coord_rot(coord,R_euler)
coord_rot(dipm,R_euler)

rotM_Z = rot_Z(1/2*np.pi)

gamma_list = np.arange(0.5,2.04,0.1)

coord_dist = coord.copy()
for i in gamma_list:
    file_save_path = f'tests/coord_only/{molecule}//{molecule}_{round(i,3)}/'
    if os.path.exists(file_save_path):
        shutil.rmtree(file_save_path)

    os.mkdir(file_save_path)

    file_save_path = f'tests/coord_only/{molecule}/{molecule}_{round(i,3)}/init_coord/'
    if os.path.exists(file_save_path):
        shutil.rmtree(file_save_path)

    os.mkdir(file_save_path)

    gamma = i
    A = 0
    B = 1

    coord_dist.iloc[B,1:] = (1-gamma) * coord.iloc[A,1:] + gamma * coord.iloc[B,1:]

    file_save_path_c = file_save_path + f'coord.xyz'

    if os.path.exists(file_save_path):
        shutil.rmtree(file_save_path)

    os.mkdir(file_save_path)


    f = open(file_save_path + f'coord.xyz',"w")

    f.write(head[0])
    f.write(head[1])
    f.close()
    coord_dist.to_csv(file_save_path +'coord.xyz', mode ='a',sep = '\t',header = None , index = False)




In [40]:
molecule = 'H2O_rot'
input_path_coord = f'tests/coord_only/{molecule}/rot0/coord.xyz'

coord,head = import_coord(input_path_coord)

rotM = rot_Z(90/360*2*np.pi)

for i in ['rot90','rot180','rot270']:
    coord_rot(coord,rotM)
    i = f'tests/coord_only/{molecule}/' + i
    if os.path.exists(i):
        shutil.rmtree(i)

    os.mkdir(i)

    f = open( i + f'/coord.xyz',"w")

    f.write(head[0])
    f.write(head[1])
    f.close()
    coord.to_csv(i +'/coord.xyz', mode ='a',sep = '\t',header = None , index = False)



In [5]:
angle = [0,90,180,270]
ml_feature = pd.DataFrame()

for i in range(4):
    input_path_coord = f'tests/H2O_rot/rot{angle[i]}/coord.xyz'
    input_path_features = f'tests/H2O_rot/rot{angle[i]}/ml_feature.csv'

    import_coord(input_path_coord)

    ml_feature[2*i] = (pd.read_csv(input_path_features).iloc[0]).T
    ml_feature[2*i+1] = (pd.read_csv(input_path_features).iloc[1]).T

qm_atom = ml_feature.loc[('qm_atom_xx','qm_atom_yy', 'qm_atom_zz','qm_atom_xy','qm_atom_xz','qm_atom_yz'),:].T

qm_delta = ml_feature.loc[('qm_delta_xx','qm_delta_yy', 'qm_delta_zz','qm_delta_xy','qm_delta_xz','qm_delta_yz'),:].T


    #Transform to vector by QM x I

print(qm_atom)

qm_atom_mat = np.zeros([3,len(qm_atom)])

qm_delta_mat = np.zeros([3,len(qm_atom)])

I = np.array([1,1,1])

for k in range(len(qm_atom.iloc[:,0])):

    qm_atom_mat[:,k] = matmul(qm_matrix(qm_atom.iloc[k,:],'qm_atom_'),I)

    qm_delta_mat[:,k] = matmul(qm_matrix(qm_delta.iloc[k,:],'qm_delta_'),I)

print(qm_atom_mat[:,0])

print(qm_atom_mat[:,2])

print(qm_atom_mat[:,4])

print(qm_atom_mat[:,6])



   qm_atom_xx  qm_atom_yy  qm_atom_zz  qm_atom_xy  qm_atom_xz  qm_atom_yz
0   -0.062637    0.033900    0.028737        -0.0   -0.000000    0.009546
1   -0.099462   -0.075616    0.175078         0.0   -0.000000   -0.013082
2    0.033900   -0.062637    0.028737        -0.0   -0.009546    0.000000
3   -0.075616   -0.099462    0.175078        -0.0    0.013082    0.000000
4   -0.062637    0.033900    0.028737         0.0   -0.000000   -0.009546
5   -0.099462   -0.075616    0.175078         0.0   -0.000000    0.013082
6    0.033900   -0.062637    0.028737        -0.0    0.009546   -0.000000
7   -0.075616   -0.099462    0.175078        -0.0   -0.013082   -0.000000
[-0.06263674  0.04344586  0.0382824 ]
[ 0.02435435 -0.06263674  0.01919087]
[-0.06263674  0.02435434  0.01919088]
[ 0.04344586 -0.06263674  0.03828241]


In [11]:
angle = [0,90,180,270]
ml_feature = pd.DataFrame()

for i in range(1):
    input_path_coord = f'tests/H2O_rot/rot{angle[i]}/coord.xyz'
    input_path_features = f'tests/H2O_rot/rot{angle[i]}/ml_feature.csv'
    input_path_hessian = f'tests/H2O_rot/rot{angle[i]}/hessian'

    coord,head = import_coord(input_path_coord)
    hess = import_hess(input_path_hessian,coord)

rotM = np.array([[0.,1.,0.],
                [1.,0.,0.],
                [0.,0.,1.]])#rot_Z(45/360*2*np.pi)
                
lamb, rotM = linalg.eigh(hess[3:6,3:6])

s, u, v = linalg.svd(hess[3:6,3:6])

print(s)

print(u)

print(v.T)

print(matmul(matmul(np.transpose(rotM),hess[3:6,3:6]),rotM))

[[-3.30167000e-06 -5.35798141e-06 -1.00000000e+00]
 [ 3.56790213e-02 -9.99363301e-01  5.23676964e-06]
 [-9.99363301e-01 -3.56790212e-02  3.49073536e-06]]
[4.73936006e-01 4.35011270e-02 6.41521696e-12]
[[-3.30167000e-06 -5.35798142e-06  1.00000000e+00]
 [ 3.56790213e-02 -9.99363301e-01 -5.23676964e-06]
 [-9.99363301e-01 -3.56790212e-02 -3.49073536e-06]]
[[-6.41521696e-12  4.21906459e-23 -2.90168536e-22]
 [ 3.71286630e-23  4.35011270e-02 -2.02428133e-16]
 [-1.62665293e-22 -2.03440532e-16  4.73936006e-01]]


In [14]:
file_path = 'tests/H2O_for_pics/'
input_path_coord = f'{file_path}'+'start_coord/'+'coord.xyz'

input_path_hess = f'{file_path}'+'start_coord/'+'hessian'

input_path_dipm = f'{file_path}'+'start_coord/'+'xyz_dipm.csv'

coord,head = import_coord(input_path_coord)
hessian = import_hess(input_path_hess,coord)
dipm = import_dipm(input_path_dipm)

dipm = dipm.iloc[:,:-3]
s = np.array([1,2,-3])
vec_trans(coord,s)

rotMat = matmul(rot_X(np.pi*0.3),rot_Z(np.pi*0.5))

coord_rot(coord,rotMat)

print(coord)


coord_rot(dipm,rotMat)

print(dipm)


P = rotM_hess(rotMat,coord)

rot_hess = matmul(matmul(P,hessian),np.transpose(P))

file_path_inert_CS = f'{file_path}' + 'init_coord/'

if os.path.exists(file_path_inert_CS):
     shutil.rmtree(file_path_inert_CS)
os.mkdir(file_path_inert_CS)

dipm.to_csv(file_path_inert_CS+'xyz_dipm.csv', sep = '\t')

df_out = pd.DataFrame(rot_hess)

df_out.to_csv(file_path_inert_CS+'hessian',sep = '\t')

f = open(file_path_inert_CS + f'coord.xyz',"w")

f.write(head[0])
f.write(head[1])
f.close()

coord.to_csv(file_path_inert_CS +'coord.xyz', mode ='a',sep = '\t',header = None , index = False)

np.savetxt(f'{file_path}'+'P_start_init',P)

  atoms         x         y         z
0     O  2.065351 -3.014836  0.954339
1     H  1.481309 -3.463307  0.337071
2     H  1.481309 -2.566365  1.571606
   Atom Number    x_dipm        y_dipm        z_dipm
0            8 -0.169899  5.877853e-09  8.090170e-09
1            1 -0.048790 -3.246862e-02 -4.468922e-02
2            1 -0.048790  3.246862e-02  4.468923e-02


In [14]:
import pandas as pd
import numpy as np
from functions import *
import os
import shutil

molecule = 'H3'
input_path_coord = f'tests/coord_only/{molecule}/startcoord/coord.xyz'

coord,head = import_coord(input_path_coord)
coord_trans = np.linspace(0,2.8,10)
for i in range(len(coord_trans)):
    fname = f'tests/coord_only/{molecule}/{molecule}_{i}'
    if os.path.exists(fname):
        shutil.rmtree(fname)

    os.mkdir(fname)
    os.mkdir(fname + '/init_coord')
    coord.iloc[1,1] = coord_trans[i]

    f = open( fname + f'/init_coord/coord.xyz',"w")

    f.write(head[0])
    f.write(head[1])
    f.close()
    coord.to_csv(fname +'/init_coord/coord.xyz', mode ='a',sep = '\t',header = None , index = False)

In [10]:
import pandas as pd
import numpy as np
from functions import *
import os
import shutil

molecule = 'H3'
input_path_coord = f'tests/main_test/{molecule}/H3_2/init_coord/coord.xyz'
input_path_hess = f'tests/main_test/{molecule}/H3_2/init_coord/hessian'
coord,head = import_coord(input_path_coord)
hess = import_hessian(input_path_hess,coord)

R = rot_Z(np.pi)
coord = coord_rot(coord,R)
R_hess = rotM_hess(R,coord)
print(coord)
hess_R = (matmul(matmul(R_hess,hess),np.transpose(R_hess)))
d_hess = hess_R - hess
for i in range(len(hess_R)):
    print(d_hess[i])


  atoms         x             y    z
0     H  0.600000 -7.347881e-17  0.0
1     H -0.622222  7.620025e-17  0.0
2     H -3.400000  4.163799e-16  0.0
[ 0.00000000e+00  4.04529701e-18 -2.30600000e-07  0.00000000e+00
 -4.27232831e-18  2.43400000e-07  0.00000000e+00  2.25401754e-19
 -1.24000000e-08]
[ 4.04529701e-18  0.00000000e+00  1.41201776e-23 -4.27232831e-18
  0.00000000e+00 -1.49039515e-23  2.25401754e-19  0.00000000e+00
  7.59281015e-25]
[-2.30600000e-07  1.41201776e-23  0.00000000e+00  2.43400000e-07
 -1.49039515e-23  0.00000000e+00 -1.28000000e-08  7.83773951e-25
  0.00000000e+00]
[ 0.00000000e+00 -4.27232831e-18  2.43400000e-07  0.00000000e+00
  5.04963985e-18 -2.87600000e-07  0.00000000e+00 -7.73738128e-19
  4.32000000e-08]
[-4.27232831e-18  0.00000000e+00 -1.49039515e-23  5.04963985e-18
  0.00000000e+00  1.76104210e-23 -7.73738128e-19  0.00000000e+00
 -2.64523709e-24]
[ 2.43400000e-07 -1.49039515e-23  0.00000000e+00 -2.87600000e-07
  1.76104210e-23  0.00000000e+00  4.40000000e-0